In [15]:
from IPython.display import Markdown, display
import os
from tqdm import tqdm
from yandex_cloud_ml_sdk import YCloudML
from glob import glob
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
import json
from yandex_cloud_ml_sdk.search_indexes import (
    StaticIndexChunkingStrategy,
    HybridSearchIndexType,
    ReciprocalRankFusionIndexCombinationStrategy,
)

def printx(string):
    display(Markdown(string))

folder_id = 'b1gst3c7cskk2big5fqn'
api_key = 'AQVNzzJielnSayrAOlQWlxDMK49OShvzdqtUQdAp'

sdk = YCloudML(folder_id=folder_id, auth=api_key)
model = sdk.models.completions("yandexgpt", model_version="rc")

In [16]:
def create_thread():
    return sdk.threads.create(ttl_days=1, expiration_policy="static")

def create_assistant(model, tools=None):
    kwargs = {}
    if tools and len(tools) > 0:
        kwargs = {"tools": tools}
    return sdk.assistants.create(
        model, ttl_days=1, expiration_policy="since_last_active", **kwargs
    )

def upload_file():
    return sdk.files.upload('/Users/ogzeus/Downloads/data2024.json', ttl_days=1, expiration_policy="static")

In [93]:
from pydantic import BaseModel, Field
from typing import Optional


class CallOperator(BaseModel):
    def process(self, thread):
        return "В корзине находятся следующие вина:\n" + "\n".join(
            [f"{x.wine_name}, число бутылок: {x.count}" for x in carts[thread.id]]
        )
        
class Agent:
    def __init__(self, assistant=None, instruction=None, search_index=None, tools=None):

        self.thread = None

        if assistant:
            self.assistant = assistant
        else:
            if tools:
                self.tools = {x.__name__: x for x in tools}
                tools = [sdk.tools.function(x) for x in tools]
            else:
                self.tools = {}
                tools = []
            if search_index:
                tools.append(sdk.tools.search_index(search_index))
            self.assistant = create_assistant(model, tools)

        if instruction:
            self.assistant.update(instruction=instruction)

    def get_thread(self, thread=None):
        if thread is not None:
            return thread
        if self.thread == None:
            self.thread = create_thread()
        return self.thread

    def __call__(self, message, thread=None):
        thread = self.get_thread(thread)
        thread.write(message)
        run = self.assistant.run(thread)
        res = run.wait()
        if res.tool_calls:
            result = []
            for f in res.tool_calls:
                print(
                    f" + Вызываем функцию {f.function.name}, args={f.function.arguments}"
                )
                fn = self.tools[f.function.name]
                obj = fn(**f.function.arguments)
                x = obj.process(thread)
                result.append({"name": f.function.name, "content": x})
            run.submit_tool_results(result)
            #time.sleep(3)
            res = run.wait()
        return res.text

    def restart(self):
        if self.thread:
            self.thread.delete()
            self.thread = sdk.threads.create(
                name="Test", ttl_days=1, expiration_policy="static"
            )

    def done(self, delete_assistant=False):
        if self.thread:
            self.thread.delete()
        if delete_assistant:
            self.assistant.delete()

In [18]:
g = upload_file()

In [19]:
op = sdk.search_indexes.create_deferred(
    g,
    index_type=HybridSearchIndexType(
        chunking_strategy=StaticIndexChunkingStrategy(
            max_chunk_size_tokens=1000, chunk_overlap_tokens=100
        ),
        combination_strategy=ReciprocalRankFusionIndexCombinationStrategy(),
    ),
)
index = op.wait()

In [20]:
# index = op.wait()

In [94]:
assistant = create_assistant(model)
thread = create_thread()

instruction = """
тебе дается 15 json файлов в которых содержится текст сообщения
из каждых этих 15 сообщений найди вопросы или утверджения, которые относятся к вузу, к процессу обучения или поступлению, и после этого,
напиши для каждого вопроса (дополненный) ответ (в формате "Вопрос1 - ответ1"), если в сообщении находится несколько вопросов,
то напиши в формате "Вопрос1 - ответ1; вопрос2 - ответ2 ...".
если это утверждение, то просто перечисляй утверждения через ;
Если тебе не понятно ничего по сообщениям или нужно уточнение, то ничего не пиши или подумай еще раз над сообщениями
1) текст сообщения ответа должен быть дополненным только в том случае, если он очень короткий, и по ответу, надо его дополнить
2) если ты не можешь ответить на вопрос, то пропускай его и ничего не отвечай пользователю
"""
agent = Agent(
    instruction=instruction
)

In [84]:
import json
import pandas as pd
from pathlib import Path

# Путь к директории с JSON файлами
json_dir = '/Users/ogzeus/Downloads'

# Находим все JSON файлы в директории
json_files = list(Path(json_dir).glob('data2024.json'))
print(json_files)
all_data = []
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
        if isinstance(data, list):
            all_data.extend(data)
        else:
            all_data.append(data)

# Создаем DataFrame из списка словарей
# Каждый ключ словаря станет столбцом
df = pd.DataFrame(all_data)

# Выводим информацию о DataFrame
print(f"Total rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print("\nПервые 5 строк DataFrame:")
print(df.head())

[PosixPath('/Users/ogzeus/Downloads/data2024.json')]
Total rows: 63628
Columns: ['id', 'text', 'reply_to_message_id']

Первые 5 строк DataFrame:
   id                                               text  reply_to_message_id
0  81  Тут вы можете общаться о будущей студентческой...                  NaN
1  83  Тут вы можете задать вопрос директору институт...                  NaN
2  85    Тут буду анонсы мероприятий, где есть восьмёрка                  NaN
3  87  Всем добрый вечер \n\nВ эту субботу - 28 октяб...                  NaN
4  88                                              ВАЖНО                  NaN


In [85]:
df

,id,text,reply_to_message_id
0,81,Тут вы можете общаться о будущей студентческой...,NaN
1,83,Тут вы можете задать вопрос директору институт...,NaN
2,85,"Тут буду анонсы мероприятий, где есть восьмёрка",NaN
3,87,Всем добрый вечер \n\nВ эту субботу - 28 октяб...,NaN
4,88,ВАЖНО,NaN
...,...,...,...
63623,70505,Мы знаем как волнительна и тревожна для первок...,NaN
63624,70507,Мы знаем как волнительна и тревожна для первок...,NaN
63625,70509,Дорогие родители студентов 1 курса 8 института...,70507.0
63626,70510,️начинаем эфир,70509.0


In [86]:
df['answer'] = None

In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63628 entries, 0 to 63627
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   63628 non-null  int64  
 1   text                 63628 non-null  object 
 2   reply_to_message_id  33080 non-null  float64
 3   answer               0 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 1.9+ MB


In [90]:
from tqdm import tqdm


texts = df['text'].tolist()
for i in tqdm(range(0, len(texts), 15), desc="Processing batches"):
    batch = texts[i:i+15]
    thread = create_thread()
    p = ''
    for text in batch:
        p += text
    result = agent(p)
    print(result)

Processing batches:   0%|                   | 1/4242 [00:11<13:05:33, 11.11s/it]

Будет ли при приёме на институт №8 «Компьютерные науки и прикладная математика» МАИ засчитываться золотая медаль ГТО 5 степени? — Индивидуальные достижения на 2024 год приёма ещё не опубликованы, ожидаем 1 ноября. В 2023 году ГТО учитывалось, подробнее здесь пункт 3.

По направлению 01.03.00 в институте 8 проходит многопрофильный конкурс. От приёмной комиссии я получила ответ, что 1 и 2 курс студенты учатся по одному учебному плану. Но не очень понятно, когда пишется заявление на желаемое направление (из направлений 01.03.00): в августе-сентябре, и тогда решающим будут баллы ЕГЭ, или уже в конце 2 курса, тогда решающим будет успеваемость студента? Или распределение происходит как-то иначе и каким внутренним документом оно регламентируется? — После зачисления на конкурсную группу (в рассматриваемом случае 01.03.00) происходит распределение по кафедрам: 01.03.02 — 806, 01.03.04 — 802, 804, 805. Распределение происходит согласно рейтингу баллов ЕГЭ и с учётом пожеланий. В случае спорных с

Processing batches:   0%|                    | 2/4242 [00:14<8:00:51,  6.80s/it]

Когда состоится День открытых дверей Института №8 МАИ «Компьютерные науки и прикладная математика»? — День открытых дверей состоится 27 января в 11:00.
Где пройдёт День открытых дверей Института №8 МАИ «Компьютерные науки и прикладная математика»? — Волоколамское шоссе, 4к6, Большой зал Приёмной комиссии МАИ.


Processing batches:   0%|                    | 3/4242 [00:19<6:35:52,  5.60s/it]

...

В данных сообщениях нет новых вопросов, требующих ответа, однако есть утверждения, связанные с вузом и процессом обучения:
* МАИ является участником пилотного проекта по модернизации высшего образования.
* Направления института №8 предлагают базовое высшее образование сроком на 4 года.
* Учебные планы изменены в пользу увеличения времени на практическую подготовку и их модульности.


Processing batches:   0%|                   | 18/4242 [01:32<6:01:47,  5.14s/it]

Можно ли подать заявление именно на кафедру 806? — Если хочется конкретно эту кафедру и без других профилей, то можно выбрать направление 02.03.02.

Можно ли где-то найти информацию о преподавательском составе направления «Прикладная математика и информатика»? — В сообщениях нет ответа на этот вопрос.

Какой уровень нужен по физике, чтобы нормально учиться на направлении «Прикладная математика и информатика»? — В сообщениях нет конкретного ответа на этот вопрос, но упоминается, что физики в учебном плане много.

Можно ли где-то посмотреть проходные баллы в магистратуру прошлых лет? — Баллы в магистратуру нерелевантны, так как каждый год может быть сложность билета, различные студенты, следовательно, балл может изменяться и достаточно сильно.


Processing batches:   0%|                   | 18/4242 [01:34<6:09:49,  5.25s/it]


KeyboardInterrupt: 